In [23]:
!pip install reverse_geocoder
import os
import io
import zipfile
import requests
import pandas as pd
import reverse_geocoder

In [7]:
url = "https://github.com/OpenBeta/climbing-data/raw/main/curated_datasets/Curated_OpenBetaAug2020_RytherAnderson.pkl.zip"

response = requests.get(url)

In [8]:
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
file_name = zip_file.namelist()[0]  
data_file = zip_file.read(file_name)        
df = pd.read_pickle(io.BytesIO(data_file))  

print("Loaded", len(df), "climbs in total")

df = df[df["type_string"] == "boulder"]
print("Boulder problems:", len(df))

Loaded 168910 climbs in total
Boulder problems: 60105


In [9]:
new_descriptions = []
for d in df["description"]:
    text = " ".join(d)
    new_descriptions.append(text.strip())
df["description"] = new_descriptions


In [10]:
word_counts = []
for text in df["description"]:
    words = text.split()
    word_counts.append(len(words))
df["desc_words"] = word_counts

In [11]:
coordinates = []
for location in df["parent_loc"]:
    longitude = location[0]
    latitude = location[1]
    coordinates.append((latitude, longitude))

In [12]:
print("Looking up states and countries...")
results = reverse_geocoder.search(coordinates)

Looking up states and countries...
Loading formatted geocoded file...


In [13]:
country_codes = []
states = []
for r in results:
    country_codes.append(r["cc"])
    states.append(r["admin1"])

In [14]:
df["country"] = country_codes
df["state"] = states

In [15]:
region_list = []
for i in range(len(df)):
    if country_codes[i] == "US":
        region_list.append(states[i])
    elif country_codes[i] == "CA":
        region_list.append("Canada")
    elif country_codes[i] == "MX":
        region_list.append("Mexico")
    else:
        region_list.append(country_codes[i])
df["region"] = region_list

In [16]:
df = df[["route_name", "parent_sector", "region", "country",
         "Vermin", "description", "desc_words"]]

In [17]:
df.columns = ["climb", "sector", "region", "country",
              "vgrade", "description", "desc_words"]

In [18]:
df.to_csv("boulders_2020_regions.csv", index=False)

In [19]:
has_description = df[df["desc_words"] > 0]
percent = 100 * len(has_description) / len(df)
print("Problems with a description:", round(percent), "%")
print("Median description length:", has_description["desc_words"].median(), "words")
print()
print("Problems per region (top 10):")
print(df["region"].value_counts().head(10))

Problems with a description: 99 %
Median description length: 31.0 words

Problems per region (top 10):
region
California       10109
Colorado          8192
Massachusetts     3627
Utah              2913
Virginia          2880
Wisconsin         2533
Arizona           2523
New Hampshire     2224
Texas             1706
Washington        1561
Name: count, dtype: int64


In [22]:
%run load_boulders.py

No existing data file found - building it from scratch.
Loaded 168910 climbs in total
Boulder problems: 60105
Looking up states and countries...
Saved to boulders_2020_regions.csv
Problems with a description: 99 %
Median description length: 31.0 words

Problems per region (top 10):
region
AD    60105
Name: count, dtype: int64
